In [1]:
import json
import os
from datetime import datetime

from langchain_ollama import ChatOllama
from langchain.agents import create_react_agent, AgentExecutor
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_experimental.tools import PythonREPLTool

c:\Users\amjat\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ══════════════════════════════════════════════════════════════════════════════
#  LONG-TERM MEMORY  —  persists to disk across sessions
# ══════════════════════════════════════════════════════════════════════════════

MEMORY_FILE = "memory_store.json"


def load_long_term_memory() -> dict:
    """Load persisted memory from disk. Returns empty structure if first run."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    return {"preferences": {}, "past_summaries": []}


def save_long_term_memory(memory: dict):
    """Write memory dict back to disk."""
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)


# Load once at startup — all tools share this reference
long_term = load_long_term_memory()

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
#  TOOLS
# ══════════════════════════════════════════════════════════════════════════════

@tool
def save_preference(input: str) -> str:
    """
    Save a user preference or important fact for future sessions.
    Use this whenever the user shares personal information, states a preference,
    or asks you to remember something.
    Input format MUST be 'key: value'
    Examples:
      'name: Amjath'
      'favorite language: Python'
      'timezone: IST'
    """
    try:
        key, value = input.split(":", 1)
        key = key.strip()
        value = value.strip()
        long_term["preferences"][key] = value
        save_long_term_memory(long_term)
        return f"✓ Remembered — {key}: {value}"
    except ValueError:
        return "Error: Use 'key: value' format, e.g. 'name: Amjath'"


@tool
def recall_memory(input: str) -> str:
    """
    Retrieve stored preferences or past session summaries from long-term memory.
    Input: 'preferences' to see saved facts, or 'history' to see past sessions.
    """
    query = input.strip().lower()
    if query == "preferences":
        prefs = long_term.get("preferences", {})
        if not prefs:
            return "No preferences saved yet."
        return "\n".join(f"{k}: {v}" for k, v in prefs.items())
    elif query == "history":
        summaries = long_term.get("past_summaries", [])
        if not summaries:
            return "No past sessions recorded yet."
        return "\n".join(summaries[-5:])  # Last 5 sessions
    else:
        return "Use 'preferences' or 'history' as input."
    

@tool
def calculator(expression: str) -> str:
    """
    Perform mathematical calculations.
    Input should be a valid math expression.
    Example: "680000 * 890000"
    """
    try:
        # Evaluate safely
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"


tools = [
    DuckDuckGoSearchRun(),
    WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper()),
    PythonREPLTool(),
    save_preference,   # ← saves facts to disk
    recall_memory,
    calculator,        # ← reads facts from disk
]

In [4]:
#  DYNAMIC SYSTEM PROMPT  

def build_prompt_template() -> PromptTemplate:
    """
    Build a ReAct PromptTemplate that injects long-term memory at the top.
    LangChain's create_react_agent expects these variables:
      {tools}, {tool_names}, {input}, {agent_scratchpad}, {chat_history}
    """
    prefs = long_term.get("preferences", {})
    prefs_text = (
        "\n".join(f"  • {k}: {v}" for k, v in prefs.items())
        if prefs else "  None yet."
    )

    past = long_term.get("past_summaries", [])
    history_text = (
        "\n".join(f"  • {s}" for s in past[-3:])
        if past else "  None yet."
    )

    template = f"""You are a helpful AI assistant with both short-term and long-term memory.

 LONG-TERM MEMORY  (persists across sessions):
Saved Preferences:
{prefs_text}

Past Session Summaries:
{history_text}

 MEMORY RULES:
- If the user shares their name, language preference, location, or any personal
  fact → call save_preference immediately.
- Refer to saved preferences naturally in your answers (e.g., use their name).
- If the user asks "what do you know about me?" → call recall_memory.

 SHORT-TERM MEMORY  (this session only):
Recent conversation:
{{chat_history}}

 REACT FORMAT  — follow this STRICTLY :
Available tools: {{tool_names}}
Tool descriptions:
{{tools}}

Use EXACTLY this format:

Question: the input question you must answer
Thought: think step-by-step about what to do
Action: one of [{{tool_names}}]
Action Input: valid input for the chosen action
Observation: result of the action
... (repeat Thought/Action/Action Input/Observation as needed)
Thought: I now know the final answer
Final Answer: your complete answer to the original question

RULES:
- ALWAYS end with "Final Answer:"
- NEVER skip steps or mix formats
- Use tools when you need current info or computation

Begin!

Question: {{input}}
Thought: {{agent_scratchpad}}"""

    return PromptTemplate.from_template(template)

In [5]:
#  SHORT-TERM MEMORY  —  in-session sliding window

short_term = ConversationBufferWindowMemory(
    k=6,                     # Keep the last 6 human/AI exchanges
    memory_key="chat_history",
    return_messages=False,   # Return as plain string (fits PromptTemplate)
    input_key="input",
    output_key="output",
)

C:\Users\amjat\AppData\Local\Temp\ipykernel_4076\2754161744.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  short_term = ConversationBufferWindowMemory(


In [6]:
llm = ChatOllama(model="mistral", temperature=0)


def build_agent_executor() -> AgentExecutor:
    """Rebuild agent with fresh prompt (picks up latest long-term memory)."""
    prompt = build_prompt_template()
    agent = create_react_agent(llm=llm, tools=tools, prompt=prompt)
    return AgentExecutor(
        agent=agent,
        tools=tools,
        memory=short_term,
        verbose=True,
        handle_parsing_errors=True,
        max_iterations=8,
    )

In [7]:
#  SESSION SUMMARY  —  saved to long-term memory when user exits

def save_session_summary(session_inputs: list[str]):
    """Condense the session into a one-line summary and persist it."""
    if not session_inputs:
        return
    first_q = session_inputs[0][:80]  # First question, truncated
    count = len(session_inputs)
    summary = (
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M')}] "
        f"{count} question(s). First: \"{first_q}\""
    )
    long_term["past_summaries"].append(summary)
    long_term["past_summaries"] = long_term["past_summaries"][-10:]  # Keep last 10
    save_long_term_memory(long_term)
    print(f"\n  Session summary saved to {MEMORY_FILE}")

In [8]:
#  MAIN LOOP

def main():
    print("\n" + "═" * 55)
    print("   ReAct Agent  ")
    print("═" * 55)

    name = long_term["preferences"].get("name", "")
    if name:
        print(f"\n  Welcome back, {name}!")
    elif long_term["preferences"]:
        print(f"\n  Welcome back! I remember your preferences.")
    else:
        print(f"\n  Hello! I'll remember things you tell me.")

    print("  Type 'quit' / 'exit' / 'q' to end the session.\n")

    session_inputs = []

    while True:
        try:
            user_input = input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            user_input = "quit"

        if not user_input:
            continue

        if user_input.lower() in {"quit", "exit", "q"}:
            save_session_summary(session_inputs)
            greeting = f"Bye, {name}!" if name else "Bye!"
            print(f"\n  {greeting}\n")
            break

        session_inputs.append(user_input)

        try:
            # Rebuild executor each turn so long-term memory stays fresh in prompt
            agent_executor = build_agent_executor()
            result = agent_executor.invoke({"input": user_input})
            final = result.get("output", "I couldn't generate a response.")
            print(f"\n  Agent: {final}\n")

        except Exception as e:
            print(f"\n  [Error] {e}\n")


if __name__ == "__main__":
    main()


═══════════════════════════════════════════════════════
   ReAct Agent 
═══════════════════════════════════════════════════════

  Welcome back! I remember your preferences.
  Type 'quit' / 'exit' / 'q' to end the session.



> Entering new AgentExecutor chain...
 I'm glad you asked! Let me check your saved preferences.

Action: recall_memory(input: 'preferences')Invalid Format: Missing 'Action Input:' after 'Action:' Action Input: 'preferences'Invalid Format: Missing 'Action:' after 'Thought:' Action: recall_memory(input: 'preferences')Invalid Format: Missing 'Action Input:' after 'Action:' Action: recall_memory(input: 'preferences')Invalid Format: Missing 'Action Input:' after 'Action:' Action: recall_memory(input: 'preferences')Invalid Format: Missing 'Action Input:' after 'Action:' Thought: I need to use the `recall_memory` tool to check your saved preferences.
Action: recall_memory(input: 'preferences')Invalid Format: Missing 'Action Input:' after 'Action:' Action: recall_memory(

c:\Users\amjat\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_community\utilities\duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


No good DuckDuckGo Search Result was found It seems that there were no relevant results found for the current weather in Chennai using DuckDuckGo Search. Let me try another method to find this information.

Action: wikipedia
Action Input: 'Weather in Chennai'Page: Chennai
Summary: Chennai, also known as Madras (its official name until 1996), is the capital and largest city of Tamil Nadu, the southernmost state of India. It is located on the Coromandel Coast of the Bay of Bengal. According to the 2011 Indian census, Chennai is the sixth-most-populous city in India and forms the fourth-most-populous urban agglomeration. Incorporated in 1688, the Greater Chennai Corporation is the oldest municipal corporation in India and the second oldest in the world after London.
Historically, the region was part of the Chola, Pandya, Pallava and Vijayanagara kingdoms during various eras. The coastal land which then contained the fishing village Madrasapattinam, was purchased by the British East India 

In [9]:

""" What is capital of Egypt, and what is 680000 * 890000?
What is the current population of India and compare it with China?
Explain the theory of relativity in simple terms and who proposed it?
What is (987654321 * 123456789) + 567890?
What are the latest AI developments in 2026?
Who is the president of France and what is the square of their age?
Find the GDP of India and calculate 10% of it.
Compare Python and Java in terms of performance and usage.
What is the current weather in Chennai?
When was Elon Musk born and how many days old is he today?
Find the capital of Japan and calculate the distance (approx) between Chennai and that city in km.
"""

' What is capital of Egypt, and what is 680000 * 890000?\nWhat is the current population of India and compare it with China?\nExplain the theory of relativity in simple terms and who proposed it?\nWhat is (987654321 * 123456789) + 567890?\nWhat are the latest AI developments in 2026?\nWho is the president of France and what is the square of their age?\nFind the GDP of India and calculate 10% of it.\nCompare Python and Java in terms of performance and usage.\nWhat is the current weather in Chennai?\nWhen was Elon Musk born and how many days old is he today?\nFind the capital of Japan and calculate the distance (approx) between Chennai and that city in km.\n'